# Lab 2: Create a Simple AI Agent

In this lab, we'll introduce you to AI agents by creating a simple agent that will create a bar graph based on data that we give to it.

#### Step 1: Load packages

In [ ]:
import os
from typing import Any
from pathlib import Path
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import AzureCliCredential, DefaultAzureCredential
from azure.ai.agents.models import CodeInterpreterTool # previously in azure.ai.projects.models

load_dotenv() # Load environment variables from .env file


#### Step 2: Connect to your Microsoft Foundry project

In [ ]:
# Use AzureCliCredential (requires az login) for the agents API which needs token-based auth
credential = AzureCliCredential()

# Connect to Microsoft Foundry project
project = AIProjectClient(
    endpoint=os.getenv("AIPROJECT_ENDPOINT"),
    credential=credential
)


#### Step 3: Create the simple AI Agent

This step demonstrates how to use the Azure AI Agents SDK to create and interact with an AI agent that can interpret code. The process includes:
- Initializing a code interpreter tool and creating an agent with it.
- Creating a communication thread for the agent.
- Sending a message to the agent with a request to generate a bar chart from provided health plan data.
- Running the agent and monitoring the run status.
- Retrieving and displaying all messages in the thread, and downloading any generated image file.
- Cleaning up by deleting the agent after the process is complete.

This workflow shows how to automate data analysis and visualization tasks using conversational AI agents in Azure.

In [ ]:
code_interpreter = CodeInterpreterTool()
with project:
    # Create an agent with the CodeInterpreterTool
    agent = project.agents.create_agent(
        model=os.environ["CHAT_MODEL"],
        name="my-agent",  # Name of the agent
        instructions="You are a helpful agent",  # Instructions for the agent
        tools=code_interpreter.definitions,
    )
    print(f"Created agent, ID: {agent.id}")

    # Create a thread for communication
    thread = project.agents.threads.create()
    print(f"Created thread, ID: {thread.id}")
    
    # Add a message to the thread
    message = project.agents.messages.create(
        thread_id=thread.id,
        role="user",
        content=(
            "Could you please create a bar chart for the using the following data and provide the file to me? "
            "Name the file as health-plan-comparision.png.\n\n"
            "Here is the data:\n"
            "Provider        Monthly Premium    Deductible    Out-of-Pocket Limit\n"
            "Northwind       $300               $1,500        $6,000\n"
            "Aetna           $350               $1,000        $5,500\n"
            "United Health   $250               $2,000        $7,000\n"
            "Premera         $200               $2,200        $6,500\n"
        ),
    )
    print(f"Created message, ID: {message['id']}")
    
    # Create and process an agent run
    run = project.agents.runs.create_and_process(thread_id=thread.id, agent_id=agent.id)
    print(f"Run finished with status: {run.status}")
    
    # Check if the run failed
    if run.status == "failed":
        print(f"Run failed: {run.last_error}")
    
    # Fetch and log all messages
    messages = project.agents.messages.list(thread_id=thread.id)
    print("Conversation:")
    found_file = False
    for msg in messages:
        print(f"{msg.role}: {msg.content}")
        # Handle file downloads from content annotations (SDK >= 1.0.0b6 uses file_citation)
        if isinstance(msg.content, list):
            for content_part in msg.content:
                annotations = []
                if isinstance(content_part, dict):
                    annotations = content_part.get("text", {}).get("annotations", [])
                elif hasattr(content_part, "text") and hasattr(content_part.text, "annotations"):
                    annotations = content_part.text.annotations

                for annotation in annotations:
                    # Handle dict-style annotations
                    ann_type = annotation.get("type") if isinstance(annotation, dict) else getattr(annotation, "type", None)
                    if ann_type in ("file_path", "file_citation"):
                        if isinstance(annotation, dict):
                            file_id = (annotation.get("file_path") or annotation.get("file_citation") or {}).get("file_id")
                            ann_text = annotation.get("text", "file")
                        else:
                            file_ref = getattr(annotation, "file_path", None) or getattr(annotation, "file_citation", None)
                            file_id = getattr(file_ref, "file_id", None)
                            ann_text = getattr(annotation, "text", "file")
                        
                        if file_id:
                            file_name = Path(ann_text).name or "health-plan-comparision.png"
                            try:
                                file_content = project.agents.files.get_content(file_id=file_id)
                                with open(file_name, "wb") as f:
                                    for chunk in file_content:
                                        f.write(chunk)
                                print(f"Saved image file to: {Path.cwd() / file_name}")
                                found_file = True
                            except Exception as e:
                                print(f"Not able to download image from agent: {e}")

    if not found_file:
        print("No image file was generated by the agent.")
    
    # Delete the agent when done
    project.agents.delete_agent(agent.id)
    print("Deleted agent")
